# Preconditions
`./setup_auth.ipynb` and `./setup_catalog_policies.ipynb`will run

In [1]:
%run ./setup_catalog_policies.ipynb

/Users/apabook/Desktop/ds-protocol/static/tutorial/venv/bin/python
{
    "participant_id": "did:jwk:provider",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1200",
    "token": null,
    "saved_at": "2026-03-24T08:23:27.578761",
    "last_interaction": "2026-03-24T08:23:27.578823",
    "is_me": true
}
{
    "participant_id": "did:jwk:consumer",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1100",
    "token": null,
    "saved_at": "2026-03-24T08:23:27.551672",
    "last_interaction": "2026-03-24T08:23:27.551707",
    "is_me": true
}
Provider DID: did:jwk:provider

Provider token: token

Consumer DID: did:jwk:consumer

Consumer token: token
{
    "dctConformsTo": null,
    "dctCreator": null,
    "dctIdentifier": "urn:catalog:0689bcec-514f-48ca-b168-14ae10a84364",
    "dctIssued": "2026-03-24T08:23:27.656914Z",
    "dctModified": null,
    "dctTitle": null,
    "dspaceMainCatalo

# Contract Negotiation

## Initialization of negotiation request (Consumer -> Provider)

In [2]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request-init"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "18"
                }]
            }
        ]
    }
}




try:
    response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
    response_as_json = response.json()
    cn_consumer_id = response_as_json["response"]["consumerPid"]
    cn_provider_id = response_as_json["response"]["providerPid"]
    print(json.dumps(response_as_json, indent=2))
except Exception as e:
    print("Error in response, {}.".format(e))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "offer": {
      "@id": "urn:odrl-policy:b4478dbe-ff6e-4538-8870-d8aae11e79f3",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "18",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:3cb00c98-1b19-4f0d-b653-97289cb9393a"
    }
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf",
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:6347

## Provider creates initial offer (Provider -> Consumer)

In [3]:
# Provider creates initial offer (Provider -> Consumer)
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,  # remove to test offer from provider
    "providerPid": cn_provider_id,  # remove to test offer from provider
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "21"
                }]
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:b4478dbe-ff6e-4538-8870-d8aae11e79f3",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "21",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:3cb00c98-1b19-4f0d-b653-97289cb9393a"
    },
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf",
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:1c7ffd34-dc74-44b2-aa

## Consumer sends negotiation request based on offer (Consumer -> Provider)

In [4]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use"

            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:b4478dbe-ff6e-4538-8870-d8aae11e79f3",
      "permission": [
        {
          "action": "use"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:3cb00c98-1b19-4f0d-b653-97289cb9393a"
    },
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf",
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:634710aa-8e83-4516-955b-b9a9b47beb58",
    "state": "REQUESTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http

## Provider updates/confirms the offer (Provider -> Consumer)

In [5]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "supermegause"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:b4478dbe-ff6e-4538-8870-d8aae11e79f3",
      "permission": [
        {
          "action": "supermegause"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:3cb00c98-1b19-4f0d-b653-97289cb9393a"
    },
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf",
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:1c7ffd34-dc74-44b2-aada-8ec90614d507",
    "state": "OFFERED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": 

## Consumer accepts the offer (Consumer -> Provider)

In [6]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-acceptance"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf",
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "state": "ACCEPTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:634710aa-8e83-4516-955b-b9a9b47beb58",
    "state": "ACCEPTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-24T09:20:19.141299Z",
    "updatedAt": "2026-03-24T09:20:20.261183Z",
    "identifiers": {
      "providerPid": "urn:provider-pid:7

## Provider creates the Agreement (Provider -> Consumer)

In [7]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-agreement"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf",
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "state": "AGREED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:1c7ffd34-dc74-44b2-aada-8ec90614d507",
    "state": "AGREED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-24T09:20:19.062951Z",
    "updatedAt": "2026-03-24T09:20:20.516449Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid:b5150

## Consumer verifies the agreement (Consumer -> Provider)

In [8]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-verification"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf",
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "state": "VERIFIED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:634710aa-8e83-4516-955b-b9a9b47beb58",
    "state": "VERIFIED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-24T09:20:19.141299Z",
    "updatedAt": "2026-03-24T09:20:21.484215Z",
    "identifiers": {
      "providerPid": "urn:provider-pid:7

## Provider finalizes the negotiation (Provider -> Consumer)

In [9]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-finalization"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    agreement = response_as_json
    agreement_id = response_as_json["negotiationAgentModel"]["agreement"]["id"]
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:b5150fea-ce7b-40df-8bbb-f940d08659bf",
    "providerPid": "urn:provider-pid:7d0298a9-2aed-4b37-8e79-1b085b14613f",
    "state": "FINALIZED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:1c7ffd34-dc74-44b2-aada-8ec90614d507",
    "state": "FINALIZED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-24T09:20:19.062951Z",
    "updatedAt": "2026-03-24T09:20:22.192471Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid

## Final agreement

In [10]:
print("Final agreement: \n{}\n".format(json.dumps(agreement["negotiationAgentModel"]["agreement"], indent=2)))
print("Final agreement id: \n{}\n".format(agreement_id))

Final agreement: 
{
  "id": "urn:agreement:043a1985-ac1d-4fd8-8352-97feec3759fc",
  "negotiationAgentProcessId": "urn:negotiation-process:1c7ffd34-dc74-44b2-aada-8ec90614d507",
  "negotiationAgentMessageId": "urn:negotiation-message:32e1c71c-0bcd-414a-b685-cfff9cce53c3",
  "consumerParticipantId": "did:jwk:consumer",
  "providerParticipantId": "did:jwk:provider",
  "agreementContent": {
    "@id": "urn:agreement:043a1985-ac1d-4fd8-8352-97feec3759fc",
    "@type": "Agreement",
    "assignee": "did:jwk:consumer",
    "assigner": "did:jwk:provider",
    "permission": [
      {
        "action": "supermegause"
      }
    ],
    "target": "urn:dataset:3cb00c98-1b19-4f0d-b653-97289cb9393a",
    "timestamp": "1774344020"
  },
  "target": "urn:dataset:3cb00c98-1b19-4f0d-b653-97289cb9393a",
  "state": "ACTIVE",
  "createdAt": "2026-03-24T09:20:20.528048Z",
  "updatedAt": "2026-03-24T09:20:22.199390Z"
}

Final agreement id: 
urn:agreement:043a1985-ac1d-4fd8-8352-97feec3759fc



# Transfer Negotiation with Agents


## Transfer request setup (Consumer -> Provider)


In [36]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-request"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+pushing",
    "dataAddress": {
        "@type": "DataAddress",
        "endpointType": "https://w3id.org/idsa/v4.1/HTTP",
        "endpoint": "http://127.0.0.1:1112",
        "endpointProperties": [
            {
                "@type": "EndpointProperty",
                "name": "authorization",
                "value": "TOKEN-ABCDEFG"
            },
            {
                "@type": "EndpointProperty",
                "name": "authType",
                "value": "bearer"
            }
        ]
    },
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_process_consumer_pid = response_as_json["response"]["consumerPid"]
transfer_process_provider_pid = response_as_json["response"]["providerPid"]
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "agreementId": "urn:agreement:043a1985-ac1d-4fd8-8352-97feec3759fc",
    "format": "http+pushing",
    "dataAddress": {
      "endpointType": "https://w3id.org/idsa/v4.1/HTTP",
      "endpoint": "http://127.0.0.1:1112",
      "endpointProperties": [
        {
          "name": "authorization",
          "value": "TOKEN-ABCDEFG"
        },
        {
          "name": "authType",
          "value": "bearer"
        }
      ]
    },
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:41379532-ffa9-4472-8c86-97bf45d11e76",
    "providerPid": "urn:provider-pid:afcd30fd-0d69-4965-86e2-05f3a1fedcc6",
    "state": "REQUESTED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:e9e7ab1

## Start transfer (Provider -> Consumer)

In [37]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:41379532-ffa9-4472-8c86-97bf45d11e76",
    "providerPid": "urn:provider-pid:afcd30fd-0d69-4965-86e2-05f3a1fedcc6"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:41379532-ffa9-4472-8c86-97bf45d11e76",
    "providerPid": "urn:provider-pid:afcd30fd-0d69-4965-86e2-05f3a1fedcc6",
    "state": "STARTED",
    "dataAddress": {
      "endpointType": "HttpProxy",
      "endpoint": "http://127.0.0.1:1100/dataplane/proxy/urn:dataplane-transfer:9d5b53cb-147b-4c1c-b808-026d23be8d3e",
      "endpointProperties": null
    }
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:f00640d8-5cd4-4b00-85f7-375b7907b083",
    "state": "STARTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "connectorInstanceId": "http+pushing",
    "transferDirection": "Push",
   

## Suspend transfer (Consumer -> Provider)

In [38]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:41379532-ffa9-4472-8c86-97bf45d11e76",
    "providerPid": "urn:provider-pid:afcd30fd-0d69-4965-86e2-05f3a1fedcc6",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:41379532-ffa9-4472-8c86-97bf45d11e76",
    "providerPid": "urn:provider-pid:afcd30fd-0d69-4965-86e2-05f3a1fedcc6",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:e9e7ab12-2c4f-4558-82ed-a0e4eaefe920",
    "state": "SUSPENDED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "connectorInstanceId": "http+pushing",
    "transferDirection": "Push",
    "agreementId": "urn:agreement:043a1985-ac1d-4fd8-8352-97feec3759fc",
    "callbackAddress": "http://127.0.0.1:1200/ds

## Restart transfer (Consumer -> Provider)

In [39]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:41379532-ffa9-4472-8c86-97bf45d11e76",
    "providerPid": "urn:provider-pid:afcd30fd-0d69-4965-86e2-05f3a1fedcc6"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:41379532-ffa9-4472-8c86-97bf45d11e76",
    "providerPid": "urn:provider-pid:afcd30fd-0d69-4965-86e2-05f3a1fedcc6",
    "state": "STARTED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:e9e7ab12-2c4f-4558-82ed-a0e4eaefe920",
    "state": "STARTED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "connectorInstanceId": "http+pushing",
    "transferDirection": "Push",
    "agreementId": "urn:agreement:043a1985-ac1d-4fd8-8352-97feec3759fc",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "created

## Suspension by Provider (Provider -> Consumer)
(Note: Testing provider-initiated suspension)

In [40]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:41379532-ffa9-4472-8c86-97bf45d11e76",
    "providerPid": "urn:provider-pid:afcd30fd-0d69-4965-86e2-05f3a1fedcc6",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:41379532-ffa9-4472-8c86-97bf45d11e76",
    "providerPid": "urn:provider-pid:afcd30fd-0d69-4965-86e2-05f3a1fedcc6",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:f00640d8-5cd4-4b00-85f7-375b7907b083",
    "state": "SUSPENDED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "connectorInstanceId": "http+pushing",
    "transferDirection": "Push",
    "agreementId": "urn:agreement:043a1985-ac1d-4fd8-8352-97feec3759fc",
    "callbackAddress": "http://127.0.0.1:1100/ds

## Failure Test: Attempt start with invalid parameters

In [41]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "dataAddress": None
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "message": "Invalid Format",
  "error_code": 3120,
  "details": "Failed to deserialize the JSON body into the target type: dataAddress: unknown field `dataAddress`, expected `consumerPid` or `providerPid` at line 1 column 158",
  "cause": "Failed to deserialize the JSON body into the target type: dataAddress: unknown field `dataAddress`, expected `consumerPid` or `providerPid` at line 1 column 158"
}


## Failure Test: Attempt duplicate or invalid suspension

In [42]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:41379532-ffa9-4472-8c86-97bf45d11e76",
    "providerPid": "urn:provider-pid:afcd30fd-0d69-4965-86e2-05f3a1fedcc6",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "5000",
    "reason": [
      "Parse Error\n"
    ]
  }
}


## Finalize transfer (Provider -> Consumer)

In [43]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-completion"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:41379532-ffa9-4472-8c86-97bf45d11e76",
    "providerPid": "urn:provider-pid:afcd30fd-0d69-4965-86e2-05f3a1fedcc6"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:41379532-ffa9-4472-8c86-97bf45d11e76",
    "providerPid": "urn:provider-pid:afcd30fd-0d69-4965-86e2-05f3a1fedcc6",
    "state": "COMPLETED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:f00640d8-5cd4-4b00-85f7-375b7907b083",
    "state": "COMPLETED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "connectorInstanceId": "http+pushing",
    "transferDirection": "Push",
    "agreementId": "urn:agreement:043a1985-ac1d-4fd8-8352-97feec3759fc",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "cre